In [ ]:
!pip install POT

In [ ]:
from scipy.ndimage import gaussian_filter
import cv2
import matplotlib.cm as cm

from scipy.stats import pearsonr
import ot
import numpy as np

import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import os
import pandas as pd


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Human heatmap

In [ ]:
def set_values():
  screen_width, screen_height=1920,1080  # 4:3
  meme_width, meme_height=768,768

  offset_x = (screen_width - meme_width) // 2
  offset_y = (screen_height - meme_height) // 2

  heatmap = np.zeros((meme_height, meme_width), dtype=float)
  return offset_x, offset_y, heatmap

In [ ]:
def set_heatmap(df, offset_x, offset_y, base_heatmap):
  heatmap=base_heatmap.copy()
  for _, row in df.iterrows():
      x = int(round(row['CURRENT_FIX_X'] - offset_x))
      y = int(round(row['CURRENT_FIX_Y'] - offset_y))
      duration = row['CURRENT_FIX_DURATION']
      if 0 <= x < 768 and 0 <= y < 768:
        heatmap[y,x] += duration   #inverse because of the matrix [row=height, column=width]

  heatmap = gaussian_filter(heatmap, sigma=20)  # to blur the image
  return heatmap

# Evaluation

In [ ]:
def NSS(pred_map, gt_fix):
    """Normalized Scanpath Saliency"""
    pred_z = (pred_map - np.mean(pred_map)) / (np.std(pred_map) + 1e-8)
    return np.mean(pred_z[gt_fix > 0])


def EMD_2d(pred_map, gt_map, size=64):
    """Earth Mover's Distance"""
    # resize of the two maps
    cam_small = cv2.resize(pred_map, (size, size))
    gt_map_small = cv2.resize(gt_map, (size, size))

    # Flatten -> vectors
    a = cam_small.ravel().astype(np.float64)
    b = gt_map_small.ravel().astype(np.float64)

    # Normalization on probability distribution
    a /= (a.sum() + 1e-8)
    b /= (b.sum() + 1e-8)

    # matrix of coordinates (pixel)
    x, y = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    coords = np.stack([x.ravel(), y.ravel()], axis=1)

    # Cost matrix = euclidean distance in pixel
    M = ot.dist(coords, coords, metric="euclidean")

    # EMD = minimal cost to transform a in b
    return ot.emd2(a, b, M)


# Main loop

In [ ]:
import pandas as pd
text_df=pd.read_csv('./TRASCRIZIONI.csv')

In [ ]:
texts = []
ids = []
for i in range(len(text_df)):
        #doc_id= text_df["file_name"][i]
        #text = text_df["full_text"][i]
        doc_id= text_df["MEME"][i]
        text = text_df["TEXT"][i]
        ids.append(doc_id)
        texts.append(text)

# Last Layer

In [ ]:
results = []

for doc_id, text in zip(ids, texts):
    print(f"[INFO] Elaboro: {doc_id}")

    meme_aggregated = pd.read_csv(f'drive/MyDrive/LM-Info-Pizzo-Davide/Dati/DF_MEME_AGGREGATED/{doc_id}.csv')
    offset_x, offset_y, base_heatmap = set_values()
    heatmap = set_heatmap(meme_aggregated, offset_x, offset_y, base_heatmap)


    meme_name = doc_id.replace(".jpg", "")
    cam = np.load(f"drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-11] fine_grained/{meme_name}_gradcam_and_ig.npy")

    gt_fix = np.zeros_like(cam)
    for _, row in meme_aggregated.iterrows():
        x = int(round(row['CURRENT_FIX_X'] - offset_x))
        y = int(round(row['CURRENT_FIX_Y'] - offset_y))
        if 0 <= x < gt_fix.shape[1] and 0 <= y < gt_fix.shape[0]:
            gt_fix[y, x] = 1

    # GT (heatmap smoothed) normalization [0,1]
    gt_map = heatmap
    gt_map = (gt_map - gt_map.min()) / (gt_map.max() + 1e-8)

    # Metrics
    nss = NSS(cam, gt_fix)
    emd = EMD_2d(cam, gt_map)

    # Add to results
    results.append({
        'meme_name': doc_id,
        'NSS': nss,

        'EMD': emd
    })


df_metrics = pd.DataFrame(results)
df_metrics.to_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-11] fine_grained/model_vs_human_metrics_LAST.csv", index=False)
print("Tutte le metriche salvate in model_vs_human_metrics.csv")


# Graphs

In [ ]:
output_dir = "./METRICS_PLOTS"
os.makedirs(output_dir, exist_ok=True)
df_metrics = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] fine_grained/model_vs_human_metrics_LAST.csv")

color = 'plum'

for col in ['NSS', 'EMD']:
    plt.figure(figsize=(8,5))
    plt.hist(df_metrics[col], bins=30, color=color, edgecolor='black', alpha=0.7)
    plt.title(f"Distribution of {col} for Fixed Features Classifier Finetune [-8]", fontsize=16, weight='bold')
    plt.xlabel(col, fontsize=14)
    plt.ylabel("Frequency", fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"{col}_last_finetune.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
df_metrics = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] fine_grained/model_vs_human_metrics_LAST.csv")

metrics = ['NSS', 'EMD']

for col in metrics:
    print("Fixed Features Classifier")
    #print("Full")
    print(f"\n Top 10 for {col}")
    display(df_metrics.nlargest(10, col)[['meme_name', col]])

    print(f"\n Bottom 10 for {col}")
    display(df_metrics.nsmallest(10, col)[['meme_name', col]])

## Full


In [ ]:
results = []

for doc_id, text in zip(ids, texts):
    print(f"[INFO] Elaboro: {doc_id}")

    meme_aggregated = pd.read_csv(f'drive/MyDrive/LM-Info-Pizzo-Davide/Dati/DF_MEME_AGGREGATED/{doc_id}.csv')
    offset_x, offset_y, base_heatmap = set_values()
    heatmap = set_heatmap(meme_aggregated, offset_x, offset_y, base_heatmap)


    meme_name = doc_id.replace(".jpg", "")
    #cam = np.load(f"drive/MyDrive/LM-Info-Pizzo-Davide/graphs/MODEL_SALIENCY_MAPS/SALIENCY_MAPS_FULL_FINETUNED/{meme_name}_gradcam_and_ig.npy")
    cam = np.load(f"drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-11] full_fine/{meme_name}_gradcam_and_ig.npy")
    gt_fix = np.zeros_like(cam)
    for _, row in meme_aggregated.iterrows():
        x = int(round(row['CURRENT_FIX_X'] - offset_x))
        y = int(round(row['CURRENT_FIX_Y'] - offset_y))
        if 0 <= x < gt_fix.shape[1] and 0 <= y < gt_fix.shape[0]:
            gt_fix[y, x] = 1


    gt_map = heatmap
    gt_map = (gt_map - gt_map.min()) / (gt_map.max() + 1e-8)

    nss = NSS(cam, gt_fix)
    emd = EMD_2d(cam, gt_map)

    # Aggiungi ai risultati
    results.append({
        'meme_name': doc_id,
        'NSS': nss,

        'EMD': emd
    })


df_metrics = pd.DataFrame(results)
df_metrics.to_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-11] full_fine/model_vs_human_metrics_FULL.csv", index=False)
print("Tutte le metriche salvate in model_vs_human_metrics.csv")


In [ ]:

output_dir = "drive/MyDrive/LM-Info-Pizzo-Davide/graphs/MODEL_SALIENCY_MAPS/METRICS_PLOTS"

os.makedirs(output_dir, exist_ok=True)

df_metrics = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] full_fine/model_vs_human_metrics_FULL.csv")

color = 'gold'

for col in ['NSS', 'EMD']:
    plt.figure(figsize=(8,5))
    plt.hist(df_metrics[col], bins=30, color=color, edgecolor='black', alpha=0.7)
    plt.title(f"Distribution of {col} for Full Finetune[-8]", fontsize=16, weight='bold')
    plt.xlabel(col, fontsize=14)
    plt.ylabel("Frequency", fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"{col}_full_finetune.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()


In [ ]:
metrics = ['NSS', 'EMD']

for col in metrics:
    print("Full")
    print(f"\n Top 10 for {col}")
    display(df_metrics.nlargest(10, col)[['meme_name', col]])

    print(f"\n Bottom 10 for {col}")
    display(df_metrics.nsmallest(10, col)[['meme_name', col]])

# Models comparison

In [ ]:
metrics = ['NSS', 'EMD']

df_last= pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-2] fine_grained/model_vs_human_metrics_LAST.csv")
df_full = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-2] full_fine/model_vs_human_metrics_FULL.csv")

for col in metrics:
    print(f"\n--- {col} ---")
    print("Fixed Features Classifier:")
    print(df_last[col].describe())
    print("\nFull Finetune:")
    print(df_full[col].describe())


In [ ]:

output_dir = "drive/MyDrive/LM-Info-Pizzo-Davide/graphs/MODEL_SALIENCY_MAPS/METRICS_PLOTS"

os.makedirs(output_dir, exist_ok=True)

df_last = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] fine_grained/model_vs_human_metrics_LAST.csv")
df_full = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] full_fine/model_vs_human_metrics_FULL.csv")

metrics = ['NSS', 'EMD']
colors = {'last': 'plum', 'full': 'gold'}

for col in metrics:
    plt.figure(figsize=(8,5))

    plt.hist(df_last[col], bins=30, color=colors['last'], edgecolor='black',
             alpha=0.6, label="Last Layer Finetune")

    plt.hist(df_full[col], bins=30, color=colors['full'], edgecolor='black',
             alpha=0.6, label="Full Finetune")

    plt.title(f"Comparison of {col}", fontsize=16, weight='bold')
    plt.xlabel(col, fontsize=14)
    plt.ylabel("Frequency", fontsize=14)
    plt.legend(fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()

    # Salva il grafico come immagine
    filename = os.path.join(output_dir, f"{col}_comparison.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os

output_dir = "drive/MyDrive/LM-Info-Pizzo-Davide/graphs/MODEL_SALIENCY_MAPS/METRICS_PLOTS"

os.makedirs(output_dir, exist_ok=True)

df_last = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] fine_grained/model_vs_human_metrics_LAST.csv")
df_full = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] full_fine/model_vs_human_metrics_FULL.csv")


df_last['Model'] = 'Fixed Features Classifier'
df_full['Model'] = 'Full Finetune'
df_all = pd.concat([df_last, df_full])

metrics = ['NSS','EMD']
colors = ['plum', 'gold']

for col in metrics:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df_all, x='Model', y=col, hue='Model', palette=colors)
    plt.title(f"Distribution of {col} for Both Models", fontsize=16, weight='bold')
    plt.ylabel(col, fontsize=14)
    plt.xlabel("Model", fontsize=14)
    plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()

    filename = os.path.join(output_dir, f"{col}_boxplot_comparison.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
df_last = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/Dati/MODEL_SALIENCY_MAPS/model_vs_human_metrics_FULL.csv")  # baseline Claudia
df_new  = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] full_fine/model_vs_human_metrics_FULL.csv")  # tuo

for col in ['NSS', 'EMD']:
    print(f"\n--- {col} ---")
    print("Layer -1 (baseline):")
    print(df_last[col].describe())
    print("\nLayer -2 (tuo):")
    print(df_new[col].describe())

In [ ]:
df_last = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/Dati/MODEL_SALIENCY_MAPS/model_vs_human_metrics_LAST.csv")  # baseline Claudia
df_new  = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-8] fine_grained/model_vs_human_metrics_LAST.csv")  # tuo

for col in ['NSS', 'EMD']:
    print(f"\n--- {col} ---")
    print("Layer -1 (baseline):")
    print(df_last[col].describe())
    print("\nLayer -2 (tuo):")
    print(df_new[col].describe())

### BOXPLOT GENERALE PER TUTTI I LAYER (Pizzo Davide)


In [ ]:
import seaborn as sns
base_folder = '/content/drive/MyDrive/LM-Info-Pizzo-Davide/output'

df_full_list = []
df_last_list = []

for layer_idx in range(1, 13):  #layer da 1 a 12
    layer_name = f"-{layer_idx}"

    # Full Fine
    path_full = os.path.join(base_folder, f"layer[{layer_name}] full_fine", "model_vs_human_metrics_FULL.csv")
    if os.path.exists(path_full):
        df = pd.read_csv(path_full)
        df['layer'] = layer_name
        df_full_list.append(df)
    else:
        print(f"File non trovato: {path_full}")

    # Last Layer Fine-Tuned
    path_last = os.path.join(base_folder, f"layer[{layer_name}] fine_grained", "model_vs_human_metrics_LAST.csv")
    if os.path.exists(path_last):
        df = pd.read_csv(path_last)
        df['layer'] = layer_name
        df_last_list.append(df)
    else:
        print(f"File non trovato: {path_last}")

df_full = pd.concat(df_full_list, ignore_index=True)
df_last = pd.concat(df_last_list, ignore_index=True)

# layer su asse X
layer_order = [f"-{i}" for i in range(12, 0, -1)]

output_dir = os.path.join(base_folder, "BOXPLOTS")
os.makedirs(output_dir, exist_ok=True)

color_full = 'gold'
color_last = 'plum'


# NSS Full Fine tuned
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_full, x='layer', y='NSS', order=layer_order, color=color_full)
# plotta media per ogni layer
means = df_full.groupby('layer')['NSS'].mean()
x_positions = range(len(layer_order))
plt.plot(x_positions, [means[l] for l in layer_order],
         color='red', marker='D', linewidth=2,
         markersize=6, label='Media', zorder=5)
plt.legend()
plt.title("NSS — Full Fine-Tuned per layer", fontsize=16, fontweight='bold')
plt.xlabel("Layer", fontsize=14)
plt.ylabel("NSS", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "NSS_full_finetune.png"), dpi=300)
plt.show()

# NSS Last Layer fine tuned
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_last, x='layer', y='NSS', order=layer_order, color=color_last)
# plotta media come prima
means = df_last.groupby('layer')['NSS'].mean()
x_positions = range(len(layer_order))
plt.plot(x_positions, [means[l] for l in layer_order],
         color='red', marker='D', linewidth=2,
         markersize=6, label='Media', zorder=5)
plt.legend()
plt.title("NSS — Last Layer Fine-Tuned per layer", fontsize=16, fontweight='bold')
plt.xlabel("Layer", fontsize=14)
plt.ylabel("NSS", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "NSS_last_finetune.png"), dpi=300)
plt.show()

# EMD full fine
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_full, x='layer', y='EMD', order=layer_order, color=color_full)
# plotta media come prima
means = df_full.groupby('layer')['EMD'].mean()
x_positions = range(len(layer_order))
plt.plot(x_positions, [means[l] for l in layer_order],
         color='red', marker='D', linewidth=2,
         markersize=6, label='Media', zorder=5)
plt.legend()
plt.title("EMD — Full Fine-Tuned per layer", fontsize=16, fontweight='bold')
plt.xlabel("Layer", fontsize=14)
plt.ylabel("EMD", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "EMD_full_finetune.png"), dpi=300)
plt.show()

# EMD lastr layer fine tuned
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_last, x='layer', y='EMD', order=layer_order, color=color_last)
# plotta media come prima
means = df_last.groupby('layer')['EMD'].mean()
x_positions = range(len(layer_order))
plt.plot(x_positions, [means[l] for l in layer_order],
         color='red', marker='D', linewidth=2,
         markersize=6, label='Media', zorder=5)
plt.legend()
plt.title("EMD — Last Layer Fine-Tuned per layer", fontsize=16, fontweight='bold')
plt.xlabel("Layer", fontsize=14)
plt.ylabel("EMD", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "EMD_last_finetune.png"), dpi=300)
plt.show()

print("Boxplot salvati in:", output_dir)

### Migliori e peggiori meme in base alle metroche in base al layer scelto

In [ ]:
df_metrics = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-5] fine_grained/model_vs_human_metrics_LAST.csv")

metrics = ['NSS', 'EMD']

for col in metrics:
    print("Fixed Features Classifier")
    print(f"\n Top 5 for {col}")
    display(df_metrics.nlargest(5, col)[['meme_name', col]])

    print(f"\n Bottom 5 for {col}")
    display(df_metrics.nsmallest(5, col)[['meme_name', col]])

In [ ]:
df_metrics = pd.read_csv("drive/MyDrive/LM-Info-Pizzo-Davide/output/layer[-5] full_fine/model_vs_human_metrics_FULL.csv")

metrics = ['NSS', 'EMD']

for col in metrics:
    print("Full fine Classifier")
    print(f"\n Top 5 for {col}")
    display(df_metrics.nlargest(5, col)[['meme_name', col]])

    print(f"\n Bottom 5 for {col}")
    display(df_metrics.nsmallest(5, col)[['meme_name', col]])